# Consume data from `Kafka`

- Phải chạy `./src/utils/produce_data.py` trước để hình thành topic trên `Kafka`

In [2]:
import json
import signal
from datetime import datetime, timezone

from confluent_kafka import Consumer, KafkaError, KafkaException
from confluent_kafka.schema_registry import SchemaRegistryClient
from confluent_kafka.schema_registry.avro import AvroDeserializer
from confluent_kafka.serialization import SerializationContext, MessageField

In [3]:
TOPIC = "kafka.public.nyc_taxi"
SCHEMA_REGISTRY_URL = "http://localhost:8081"

CONSUMER_CONFIG = {
    "bootstrap.servers": "localhost:9092",
    "group.id": "sensor-readings-consumer",
    # change to "latest" to see only new events
    # change to "earliest" to read entire message
    "auto.offset.reset": "earliest",
    "enable.auto.commit": True,
}

# Map Debezium operation codes to human-readable labels
OP_LABELS = {
    "c": "INSERT",
    "u": "UPDATE",
    "d": "DELETE",
    "r": "READ (snapshot)",
}

In [4]:
# Avro deserializer — fetches schema from Registry by schema_id embedded in message
_schema_registry_client = SchemaRegistryClient({"url": SCHEMA_REGISTRY_URL})
_avro_deserializer = AvroDeserializer(_schema_registry_client)

In [5]:
def _parse_value(raw: bytes | None, topic: str) -> dict | None:
    if raw is None:
        return None
    try:
        ctx = SerializationContext(topic, MessageField.VALUE)
        return _avro_deserializer(raw, ctx)
    except Exception as e:
        print(f"[consumer] Deserialize error: {e}")
        return None


def _format_record(payload: dict) -> str:
    op = payload.get("op", "?")
    label = OP_LABELS.get(op, op)
    ts_ms = payload.get("ts_ms")
    event_time = (
        datetime.fromtimestamp(ts_ms / 1000, tz=timezone.utc).isoformat()
        if ts_ms is not None
        else "unknown"
    )

    before = payload.get("before")
    after = payload.get("after")

    lines = [
        f"  Operation : {label}",
        f"  Event time: {event_time}",
    ]

    if before:
        lines.append(f"  Before    : {json.dumps(before, ensure_ascii=False)}")
    if after:
        lines.append(f"  After     : {json.dumps(after, ensure_ascii=False)}")

    return "\n".join(lines)

In [6]:
def run_consumer() -> None:
    consumer = Consumer(CONSUMER_CONFIG)
    consumer.subscribe([TOPIC])

    print(f"[consumer] Subscribed to topic '{TOPIC}'. Waiting for events... (Ctrl+C to stop)\n")

    # Graceful shutdown on SIGINT / SIGTERM
    running = True

    def _stop(signum, frame):
        nonlocal running
        print("\n[consumer] Shutting down...")
        running = False

    signal.signal(signal.SIGINT, _stop)
    signal.signal(signal.SIGTERM, _stop)

    try:
        while running:
            msg = consumer.poll(timeout=1.0)

            if msg is None:
                continue

            if msg.error():
                if msg.error().code() == KafkaError._PARTITION_EOF:
                    # End of partition — not an error, just caught up
                    continue
                raise KafkaException(msg.error())

            value = _parse_value(msg.value(), msg.topic())
            if value is None:
                print("[consumer] Failed to deserialize message, skipping.")
                continue

            payload = value.get("payload", value)  # handle wrapped or flat payloads

            print(f"[{datetime.now().isoformat()}] Change detected:")
            print(_format_record(payload))
            print()

    finally:
        consumer.close()
        print("[consumer] Connection closed.")

In [8]:
run_consumer()

[consumer] Subscribed to topic 'kafka.public.nyc_taxi'. Waiting for events... (Ctrl+C to stop)



[2026-05-04T09:00:23.614344] Change detected:
  Operation : INSERT
  Event time: 2026-05-04T08:59:32.867000+00:00
  After     : {"vendor_id": "VTS", "pickup_datetime": 1231037520000000, "dropoff_datetime": 1231038120000000, "passenger_count": 1.0, "trip_distance": 2.63, "rate_code_id": null, "store_and_fwd_flag": null, "pickup_longitude": -73.991957, "pickup_latitude": 40.721567, "dropoff_longitude": -73.993803, "dropoff_latitude": 40.695922, "pu_location_id": null, "do_location_id": null, "payment_type": "CASH", "fare_amount": 8.9, "extra": null, "mta_tax": null, "surcharge": 0.5, "improvement_surcharge": null, "tip_amount": 0.0, "tolls_amount": 0.0, "total_amount": 9.4, "congestion_surcharge": null, "airport_fee": null, "cbd_congestion_fee": null}

[2026-05-04T09:00:23.614831] Change detected:
  Operation : INSERT
  Event time: 2026-05-04T08:59:32.870000+00:00
  After     : {"vendor_id": "CMT", "pickup_datetime": 1232852994000000, "dropoff_datetime": 1232854176000000, "passenger_coun